# 27: Wu 2003 Sequential Degradation Tracking (30-Day)

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle
import time
import pathlib

from cstr_sbi.recycle.physics import (
    NOMINAL_CTRL_SB, NOMINAL_INLET, NOMINAL_Y0_EXPLICIT, PARAM_NAMES,
    simulate_trajectory_explicit, extract_observations_explicit,
    T_SP, T_J_NOM, REFLUX_RATIO, recycle_rhs_explicit, column_qss, F0_NOM, F_R_NOM
)
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SB
from cstr_sbi.recycle.scenarios import CLOSED_LOOP_NAMES, get_scenario
from cstr_sbi.recycle.simulator import nominal_warm_start
import jax.numpy as jnp

DATA = pathlib.Path('../data')
FIGURES = pathlib.Path('../figures'); FIGURES.mkdir(exist_ok=True)
SBI_LOGS = pathlib.Path('../sbi-logs'); SBI_LOGS.mkdir(exist_ok=True)
OI = ["#000000","#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7"]
print("Imports OK")

## 1. Load S-B Posterior

In [ ]:
posterior_sb = None
sb_pkl = SBI_LOGS / 'wu2003_posterior_sb.pkl'
if sb_pkl.exists():
    with open(sb_pkl, 'rb') as f:
        sb_data = pickle.load(f)
    posterior_sb = sb_data['posterior']
    print(f"Loaded S-B posterior (N_TRAIN={sb_data.get('N_TRAIN','?')})")
else:
    print("WARNING: Run nb24 first to train the S-B posterior.")

## 2. Define 30-Day Degradation Profile

In [ ]:
# 30 days = 720 two-hour windows
N_DAYS = 30
N_WINDOWS = N_DAYS * 12   # 12 windows per day (24h / 2h = 12)
print(f"Total windows: {N_WINDOWS}")

# Time vector in days (centre of each window)
t_days = np.arange(N_WINDOWS) * (2/24)  # in days

# Alpha: linear decay from 1.0 to 0.65 over 30 days
alpha_true = 1.0 - (1.0 - 0.65) * t_days / N_DAYS
alpha_true = np.clip(alpha_true, 0.65, 1.0)

# beta_r: Kern-Seaton fouling model scaled to reach ~0.90 over 30 days
t_hours = t_days * 24.0
beta_r_true = 1.0 / (1.0 + 1.5e-4 * t_hours)
beta_r_true = np.clip(beta_r_true, 0.90, 1.0)

# eta_col, xi_reb, z_A0: constant nominal
eta_col_true  = np.ones(N_WINDOWS)
xi_reb_true   = np.ones(N_WINDOWS)
z_A0_true     = np.ones(N_WINDOWS) * 0.90

theta_true = np.stack([alpha_true, beta_r_true, eta_col_true, xi_reb_true, z_A0_true], axis=1)
print(f"theta_true shape: {theta_true.shape}")  # (720, 5)
print(f"alpha range: {alpha_true.min():.3f} to {alpha_true.max():.3f}")
print(f"beta_r range: {beta_r_true.min():.3f} to {beta_r_true.max():.3f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_days, alpha_true, color=OI[1], lw=2)
axes[0].set_ylabel("alpha"); axes[0].set_title("Catalyst Decay (alpha)")
axes[0].axhline(0.85, ls='--', color='gray', alpha=0.7, label='Mild fault threshold')
axes[0].legend()
axes[1].plot(t_days, beta_r_true, color=OI[2], lw=2)
axes[1].set_ylabel("beta_r"); axes[1].set_title("Jacket Fouling (beta_r — Kern-Seaton)")
axes[1].set_xlabel("Time (days)")
plt.tight_layout()
plt.savefig(FIGURES / 'nb27_degradation_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb27_degradation_profile.png")

## 3. Generate 720 Observation Windows

In [ ]:
# Each window: 2h, 120 time steps, starting from end of previous window
print("Simulating 720 sequential observation windows...")
print("Expected time: ~5-15 minutes (includes JIT warmup)")

y0_sb = nominal_warm_start("S-B")
rng_obs = np.random.default_rng(20260701)

ctrl_arr = jnp.array(NOMINAL_CTRL_SB)
y_current = jnp.array(y0_sb)

raw_windows = []  # list of (120, 12) arrays
t_h_window = None

t0 = time.time()
for win_i in range(N_WINDOWS):
    theta_win = theta_true[win_i]
    theta_j = jnp.array(theta_win, dtype=jnp.float32)

    ts, ys = simulate_trajectory_explicit(
        theta_j, NOMINAL_INLET, ctrl_arr, y_current,
        t_final=2.0, n_save=120
    )
    raw = extract_observations_explicit(ys, theta_j, ctrl_arr)
    raw_np = np.asarray(raw)  # (120, 12)

    if t_h_window is None:
        t_h_window = np.asarray(ts)

    # Add sensor noise
    scale = np.maximum(np.max(np.abs(raw_np), axis=0), 1e-6)
    noise = rng_obs.normal(0, 0.003 * scale, raw_np.shape)
    raw_noisy = raw_np + noise

    raw_windows.append(raw_noisy)

    # Use end state as initial condition for next window
    y_current = ys[-1]

    if (win_i + 1) % 60 == 0:
        elapsed = time.time() - t0
        print(f"  Window {win_i+1}/{N_WINDOWS}  ({elapsed:.0f}s, "
              f"{elapsed/(win_i+1)*N_WINDOWS:.0f}s total est.)")

raw_windows = np.stack(raw_windows)  # (720, 120, 12)
print(f"\nSimulation complete in {time.time()-t0:.0f}s")
print(f"raw_windows shape: {raw_windows.shape}")
print(f"NaN count: {np.isnan(raw_windows).sum()}")

In [ ]:
print("Computing summary statistics for 720 windows...")
summaries_seq = np.zeros((N_WINDOWS, N_SUMMARIES_SB), dtype=np.float32)
for i in range(N_WINDOWS):
    s = compute_summaries(raw_windows[i], "S-B", t_h_window)
    if not np.isnan(s).any():
        summaries_seq[i] = s
print(f"Summaries shape: {summaries_seq.shape}")
print(f"NaN rows: {np.isnan(summaries_seq).any(axis=1).sum()}")

## 4. SBI Sequential Inference

In [ ]:
if posterior_sb is None:
    print("Skipping SBI inference — posterior not available. Run nb24 first.")
    sbi_alpha_mean = alpha_true.copy()
    sbi_alpha_lo   = alpha_true - 0.05
    sbi_alpha_hi   = alpha_true + 0.05
    sbi_beta_mean  = beta_r_true.copy()
    sbi_beta_lo    = beta_r_true - 0.02
    sbi_beta_hi    = beta_r_true + 0.02
else:
    print(f"Running SBI inference for {N_WINDOWS} windows (500 samples each)...")
    print("Expected time: ~2-5 minutes")
    sbi_alpha_mean = np.zeros(N_WINDOWS)
    sbi_alpha_lo   = np.zeros(N_WINDOWS)
    sbi_alpha_hi   = np.zeros(N_WINDOWS)
    sbi_beta_mean  = np.zeros(N_WINDOWS)
    sbi_beta_lo    = np.zeros(N_WINDOWS)
    sbi_beta_hi    = np.zeros(N_WINDOWS)

    t0 = time.time()
    for win_i in range(N_WINDOWS):
        s = summaries_seq[win_i]
        if np.isnan(s).any():
            sbi_alpha_mean[win_i] = np.nan
            sbi_beta_mean[win_i] = np.nan
            continue
        x_obs = torch.tensor(s, dtype=torch.float32)
        samp = posterior_sb.sample((500,), x=x_obs).numpy()
        sbi_alpha_mean[win_i] = samp[:, 0].mean()
        sbi_alpha_lo[win_i]   = np.percentile(samp[:, 0], 5)
        sbi_alpha_hi[win_i]   = np.percentile(samp[:, 0], 95)
        sbi_beta_mean[win_i]  = samp[:, 1].mean()
        sbi_beta_lo[win_i]    = np.percentile(samp[:, 1], 5)
        sbi_beta_hi[win_i]    = np.percentile(samp[:, 1], 95)
        if (win_i + 1) % 60 == 0:
            print(f"  {win_i+1}/{N_WINDOWS}  ({time.time()-t0:.0f}s)")

    print(f"SBI inference done in {time.time()-t0:.0f}s")

## 5. EKF Sequential Tracking

In [ ]:
def ekf_rhs_aug_np(y_aug, ctrl):
    """Augmented ODE for 9-state vector (numpy wrapper around JAX physics)."""
    z_A, T_r, T_j, I_T, R_st, V_st, alpha, beta_r, eta_col = y_aug
    xi_reb = 1.0; z_A0_eff = 0.90
    theta = jnp.array([alpha, beta_r, eta_col, xi_reb, z_A0_eff], dtype=jnp.float32)
    y8 = jnp.array([z_A, T_r, T_j, I_T, R_st, V_st, 0.0, 0.0], dtype=jnp.float32)
    ctrl_j = jnp.array(ctrl, dtype=jnp.float32)
    dy8 = np.asarray(recycle_rhs_explicit(0.0, y8, (theta, NOMINAL_INLET, ctrl_j)))
    return np.array([dy8[0], dy8[1], dy8[2], dy8[3], dy8[4], dy8[5], 0.0, 0.0, 0.0])

print("Running sequential EKF over 720 windows...")
ctrl_np = np.asarray(NOMINAL_CTRL_SB)
y0_sb_np = np.asarray(y0_sb)

x_ekf = np.array([
    float(y0_sb_np[0]), float(y0_sb_np[1]), float(y0_sb_np[2]), float(y0_sb_np[3]),
    float(y0_sb_np[4]), float(y0_sb_np[5]),
    1.0, 1.0, 1.0
])
P_ekf = np.eye(9) * 0.01
P_ekf[6, 6] = 0.05
P_ekf[7, 7] = 0.02
P_ekf[8, 8] = 0.02

Q_ekf = np.eye(9) * 1e-8
Q_ekf[6, 6] = 1e-5
Q_ekf[7, 7] = 1e-6
Q_ekf[8, 8] = 1e-8
R_ekf = np.diag([float(T_SP)*0.005, float(T_J_NOM)*0.005, 0.003])**2

obs_indices = [0, 1, 7]
eps = 1e-4
n_state = 9

ekf_alpha_seq = np.zeros(N_WINDOWS)
ekf_beta_seq  = np.zeros(N_WINDOWS)
ekf_alpha_std_seq = np.zeros(N_WINDOWS)
ekf_beta_std_seq  = np.zeros(N_WINDOWS)

t0 = time.time()
for win_i in range(N_WINDOWS):
    obs_window = raw_windows[win_i]
    dt_h = 2.0 / 120

    for k in range(len(obs_window)):
        f0 = ekf_rhs_aug_np(x_ekf, ctrl_np)
        x_pred = x_ekf + f0 * dt_h
        x_pred[0] = np.clip(x_pred[0], 1e-4, 0.999)
        x_pred[1] = max(x_pred[1], 250.0)
        x_pred[4] = np.clip(x_pred[4], 1.0, 4.0)
        x_pred[5] = np.clip(x_pred[5], 0.5, 1.8)
        x_pred[6] = np.clip(x_pred[6], 0.40, 1.20)
        x_pred[7] = np.clip(x_pred[7], 0.40, 1.20)
        x_pred[8] = np.clip(x_pred[8], 0.50, 1.00)

        F_jac = np.zeros((n_state, n_state))
        for j_idx in range(n_state):
            xp = x_ekf.copy(); xp[j_idx] += eps
            fp = ekf_rhs_aug_np(xp, ctrl_np)
            xm = x_ekf.copy(); xm[j_idx] -= eps
            fm = ekf_rhs_aug_np(xm, ctrl_np)
            F_jac[:, j_idx] = (fp - fm) / (2*eps)
        A_mat = np.eye(n_state) + F_jac * dt_h
        P_pred = A_mat @ P_ekf @ A_mat.T + Q_ekf * dt_h

        H_mat = np.zeros((3, n_state))
        H_mat[0, 1] = 1.0
        H_mat[1, 2] = 1.0
        for state_idx in [0, 4, 5, 6, 8]:
            xp = x_pred.copy(); xp[state_idx] += eps
            _, _, d_p = column_qss(float(xp[0]), float(xp[8]))
            d_p = np.clip(float(d_p), 0.01, 0.98)
            FR_p = d_p * float(F0_NOM) / (1-d_p) / float(F_R_NOM)
            xm = x_pred.copy(); xm[state_idx] -= eps
            _, _, d_m = column_qss(float(xm[0]), float(xm[8]))
            d_m = np.clip(float(d_m), 0.01, 0.98)
            FR_m = d_m * float(F0_NOM) / (1-d_m) / float(F_R_NOM)
            H_mat[2, state_idx] = (FR_p - FR_m) / (2*eps)

        _, _, d_c = column_qss(float(x_pred[0]), float(x_pred[8]))
        d_c = np.clip(float(d_c), 0.01, 0.98)
        FR_c = d_c * float(F0_NOM) / (1-d_c) / float(F_R_NOM)
        y_pred_v = np.array([x_pred[1], x_pred[2], FR_c])
        y_obs_v = obs_window[k, obs_indices]
        innov = y_obs_v - y_pred_v
        S_mat = H_mat @ P_pred @ H_mat.T + R_ekf
        try:
            K_mat = P_pred @ H_mat.T @ np.linalg.solve(S_mat, np.eye(3))
        except Exception:
            K_mat = np.zeros((n_state, 3))
        x_ekf = x_pred + K_mat @ innov
        x_ekf[0] = np.clip(x_ekf[0], 1e-4, 0.999)
        x_ekf[6] = np.clip(x_ekf[6], 0.40, 1.20)
        x_ekf[7] = np.clip(x_ekf[7], 0.40, 1.20)
        x_ekf[8] = np.clip(x_ekf[8], 0.50, 1.00)
        P_ekf = (np.eye(n_state) - K_mat @ H_mat) @ P_pred

    ekf_alpha_seq[win_i]     = x_ekf[6]
    ekf_beta_seq[win_i]      = x_ekf[7]
    ekf_alpha_std_seq[win_i] = np.sqrt(P_ekf[6, 6])
    ekf_beta_std_seq[win_i]  = np.sqrt(P_ekf[7, 7])

    if (win_i + 1) % 60 == 0:
        print(f"  Window {win_i+1}/{N_WINDOWS} | alpha_est={x_ekf[6]:.3f} | "
              f"beta_est={x_ekf[7]:.3f} | ({time.time()-t0:.0f}s)")

print(f"EKF complete in {time.time()-t0:.0f}s")

## 6. 30-Day Tracking Figure

In [ ]:
ekf_alpha_lo = ekf_alpha_seq - 1.645 * ekf_alpha_std_seq
ekf_alpha_hi = ekf_alpha_seq + 1.645 * ekf_alpha_std_seq
ekf_beta_lo  = ekf_beta_seq - 1.645 * ekf_beta_std_seq
ekf_beta_hi  = ekf_beta_seq + 1.645 * ekf_beta_std_seq

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(t_days, alpha_true, 'k-', lw=2.5, label='True alpha', zorder=5)
if not np.isnan(sbi_alpha_mean).all():
    ax.plot(t_days, sbi_alpha_mean, color=OI[2], lw=1.5, label='SBI mean')
    ax.fill_between(t_days, sbi_alpha_lo, sbi_alpha_hi, alpha=0.25, color=OI[2], label='SBI 90% CI')
ax.plot(t_days, ekf_alpha_seq, color=OI[6], lw=1.5, ls='--', label='EKF mean')
ax.fill_between(t_days, ekf_alpha_lo, ekf_alpha_hi, alpha=0.15, color=OI[6], label='EKF 90% CI')
ax.set_ylabel("alpha (catalyst activity)", fontsize=11)
ax.set_title("30-Day Sequential Tracking: SBI vs EKF", fontsize=12)
ax.legend(fontsize=9, loc='lower left')
ax.axhline(0.85, ls=':', color='gray', alpha=0.6)
ax.set_ylim(0.55, 1.05)

ax = axes[1]
ax.plot(t_days, beta_r_true, 'k-', lw=2.5, label='True beta_r', zorder=5)
if not np.isnan(sbi_beta_mean).all():
    ax.plot(t_days, sbi_beta_mean, color=OI[2], lw=1.5, label='SBI mean')
    ax.fill_between(t_days, sbi_beta_lo, sbi_beta_hi, alpha=0.25, color=OI[2])
ax.plot(t_days, ekf_beta_seq, color=OI[6], lw=1.5, ls='--', label='EKF mean')
ax.fill_between(t_days, ekf_beta_lo, ekf_beta_hi, alpha=0.15, color=OI[6])
ax.set_ylabel("beta_r (jacket HT factor)", fontsize=11)
ax.set_xlabel("Time (days)", fontsize=11)
ax.legend(fontsize=9, loc='lower left')
ax.set_ylim(0.85, 1.05)

plt.tight_layout()
plt.savefig(FIGURES / 'nb27_sequential_tracking.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb27_sequential_tracking.png")

## 7. Metrics Table

In [ ]:
import pandas as pd
metrics = []

valid = ~np.isnan(sbi_alpha_mean)
if valid.any():
    mae_sbi_alpha = np.mean(np.abs(sbi_alpha_mean[valid] - alpha_true[valid]))
    bias_sbi_alpha = np.mean(sbi_alpha_mean[valid] - alpha_true[valid])
    cov_sbi_alpha = np.mean(
        (sbi_alpha_lo[valid] <= alpha_true[valid]) & (alpha_true[valid] <= sbi_alpha_hi[valid])
    )
    metrics.append({'Param': 'alpha', 'Method': 'SBI', 'MAE': mae_sbi_alpha,
                    'Bias': bias_sbi_alpha, '90%_Coverage': cov_sbi_alpha})

mae_ekf_alpha  = np.mean(np.abs(ekf_alpha_seq - alpha_true))
bias_ekf_alpha = np.mean(ekf_alpha_seq - alpha_true)
cov_ekf_alpha  = np.mean((ekf_alpha_lo <= alpha_true) & (alpha_true <= ekf_alpha_hi))
metrics.append({'Param': 'alpha', 'Method': 'EKF', 'MAE': mae_ekf_alpha,
                'Bias': bias_ekf_alpha, '90%_Coverage': cov_ekf_alpha})

if valid.any():
    mae_sbi_beta = np.mean(np.abs(sbi_beta_mean[valid] - beta_r_true[valid]))
    bias_sbi_beta = np.mean(sbi_beta_mean[valid] - beta_r_true[valid])
    cov_sbi_beta = np.mean(
        (sbi_beta_lo[valid] <= beta_r_true[valid]) & (beta_r_true[valid] <= sbi_beta_hi[valid])
    )
    metrics.append({'Param': 'beta_r', 'Method': 'SBI', 'MAE': mae_sbi_beta,
                    'Bias': bias_sbi_beta, '90%_Coverage': cov_sbi_beta})

mae_ekf_beta  = np.mean(np.abs(ekf_beta_seq - beta_r_true))
bias_ekf_beta = np.mean(ekf_beta_seq - beta_r_true)
cov_ekf_beta  = np.mean((ekf_beta_lo <= beta_r_true) & (beta_r_true <= ekf_beta_hi))
metrics.append({'Param': 'beta_r', 'Method': 'EKF', 'MAE': mae_ekf_beta,
                'Bias': bias_ekf_beta, '90%_Coverage': cov_ekf_beta})

df_metrics = pd.DataFrame(metrics)
print("\nTracking Metrics Summary:")
print(df_metrics.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 8. Timing Comparison

In [ ]:
if posterior_sb is not None:
    s_test = compute_summaries(raw_windows[0], "S-B", t_h_window)
    x_test = torch.tensor(s_test, dtype=torch.float32)
    # Warmup
    _ = posterior_sb.sample((100,), x=x_test)
    t0 = time.time()
    n_time = 50
    for _ in range(n_time):
        _ = posterior_sb.sample((500,), x=x_test)
    sbi_ms = (time.time() - t0) / n_time * 1000
    print(f"SBI: {sbi_ms:.1f} ms per window (500 samples)")
else:
    sbi_ms = float('nan')
    print("SBI timing unavailable — posterior not loaded")

# Re-time EKF on 10 windows (simplified predict-only, no update)
x_ekf_tmp = np.array([0.5, float(T_SP), float(T_J_NOM), 0.0, float(REFLUX_RATIO), 1.0,
                       1.0, 1.0, 1.0])
t0 = time.time()
for win_i in range(10):
    obs_window = raw_windows[win_i]
    for k in range(120):
        f0 = ekf_rhs_aug_np(x_ekf_tmp, np.asarray(NOMINAL_CTRL_SB))
        x_ekf_tmp = x_ekf_tmp + f0 * (2.0/120)
        x_ekf_tmp = np.clip(x_ekf_tmp,
                             [1e-4, 250, 200, 0, 1, 0.5, 0.4, 0.4, 0.5],
                             [0.999, 450, 450, 1e6, 4, 1.8, 1.2, 1.2, 1.0])
ekf_10_time = time.time() - t0
ekf_per_window = ekf_10_time / 10
ekf_total_720 = ekf_per_window * N_WINDOWS
print(f"EKF: {ekf_per_window*1000:.1f} ms/window, {ekf_total_720:.1f}s total for {N_WINDOWS} windows")

import pandas as pd
df_timing = pd.DataFrame([
    {'Method': 'SBI (SNPE-C)',
     'ms/window': f"{sbi_ms:.0f}" if not np.isnan(sbi_ms) else "N/A",
     'Total 720-window (s)': f"{sbi_ms*720/1000:.1f}" if not np.isnan(sbi_ms) else "N/A"},
    {'Method': 'EKF (augmented)',
     'ms/window': f"{ekf_per_window*1000:.1f}",
     'Total 720-window (s)': f"{ekf_total_720:.1f}"},
])
print("\nTiming Summary:")
print(df_timing.to_string(index=False))